# Orbital Trust - experimento minimo de Machine Learning

Objetivo: demonstrar um fluxo minimo de ML conectado ao MVP Orbital Trust. O experimento usa dados mockados e controlados com o mesmo contrato IoT -> ML/API para estimar `risk_level` ambiental a partir de classes detectadas, mudanca, nuvem, sombra, qualidade da imagem e confianca CV.

Relacao com o problema: no produto, frames orbitais de Sentinel-2, Landsat, FIRMS ou INPE passam pelo pipeline CV e geram alertas de risco ambiental. Este notebook evidencia como esses campos podem alimentar um modelo simples antes de entregar a resposta para API/mobile.

Restricoes: nao usa webcam, chaves, dados pessoais, APIs privadas ou download externo. O dataset abaixo e sintetico, deterministico e serve apenas para validacao academica do fluxo.

## 1. Dataset controlado

Cada amostra segue os campos obrigatorios do payload IoT: `event_id`, `timestamp`, `area_id`, `source`, `detected_class`, `class_percentage`, `change_score`, `cloud_score`, `shadow_score`, `image_quality` e `cv_confidence`. O rotulo `risk_level` usa apenas `baixo`, `medio` ou `alto`.

In [1]:
from collections import Counter
from math import sqrt
from random import Random

RISK_LEVELS = ("baixo", "medio", "alto")
DETECTED_CLASSES = ("vegetacao", "solo_exposto", "agua", "queimada", "baixa_visibilidade")
SOURCES = ("Sentinel-2", "Landsat", "FIRMS", "INPE")


def clamp(value, minimum=0.0, maximum=1.0):
    return max(minimum, min(maximum, value))


def label_risk(detected_class, class_percentage, change_score, image_quality, cv_confidence):
    class_weight = {
        "vegetacao": 0.00,
        "agua": 0.06,
        "baixa_visibilidade": 0.10,
        "solo_exposto": 0.12,
        "queimada": 0.25,
    }[detected_class]
    extent_weight = 0.12 * (class_percentage / 100.0)
    quality_penalty = max(0.0, 0.70 - image_quality) * 0.20
    confidence_penalty = max(0.0, 0.70 - cv_confidence) * 0.15
    score = change_score + class_weight + extent_weight + quality_penalty + confidence_penalty

    if score > 0.55:
        return "alto"
    if score > 0.25:
        return "medio"
    return "baixo"


def build_mock_dataset(total=72):
    rng = Random(68068)
    rows = []
    for index in range(total):
        detected_class = DETECTED_CLASSES[index % len(DETECTED_CLASSES)]
        source = SOURCES[index % len(SOURCES)]
        base_change = {
            "vegetacao": 0.08,
            "agua": 0.14,
            "baixa_visibilidade": 0.18,
            "solo_exposto": 0.25,
            "queimada": 0.36,
        }[detected_class]
        class_percentage = round(clamp(rng.gauss(42, 18), 5, 92), 2)
        change_score = round(clamp(base_change + rng.uniform(-0.10, 0.25)), 3)
        cloud_score = round(clamp(rng.betavariate(2, 8)), 3)
        shadow_score = round(clamp(rng.betavariate(2, 10)), 3)
        image_quality = round(clamp(1.0 - (cloud_score * 0.55) - (shadow_score * 0.30)), 3)
        cv_confidence = round(clamp(image_quality - rng.uniform(0.02, 0.20)), 3)
        risk_level = label_risk(
            detected_class,
            class_percentage,
            change_score,
            image_quality,
            cv_confidence,
        )

        rows.append(
            {
                "event_id": f"EVT-MOCK-{index + 1:03d}",
                "timestamp": "2026-05-31T12:00:00Z",
                "area_id": f"BR-MT-{(index % 4) + 1:03d}",
                "source": source,
                "detected_class": detected_class,
                "class_percentage": class_percentage,
                "change_score": change_score,
                "cloud_score": cloud_score,
                "shadow_score": shadow_score,
                "image_quality": image_quality,
                "cv_confidence": cv_confidence,
                "risk_level": risk_level,
            }
        )
    return rows


dataset = build_mock_dataset()
print(f"amostras: {len(dataset)}")
print("distribuicao de risco:", dict(Counter(row["risk_level"] for row in dataset)))
dataset[:3]

amostras: 72
distribuicao de risco: {'medio': 38, 'alto': 21, 'baixo': 13}


[{'event_id': 'EVT-MOCK-001',
  'timestamp': '2026-05-31T12:00:00Z',
  'area_id': 'BR-MT-001',
  'source': 'Sentinel-2',
  'detected_class': 'vegetacao',
  'class_percentage': 90.19,
  'change_score': 0.284,
  'cloud_score': 0.251,
  'shadow_score': 0.117,
  'image_quality': 0.827,
  'cv_confidence': 0.733,
  'risk_level': 'medio'},
 {'event_id': 'EVT-MOCK-002',
  'timestamp': '2026-05-31T12:00:00Z',
  'area_id': 'BR-MT-002',
  'source': 'Landsat',
  'detected_class': 'solo_exposto',
  'class_percentage': 16.71,
  'change_score': 0.233,
  'cloud_score': 0.132,
  'shadow_score': 0.042,
  'image_quality': 0.915,
  'cv_confidence': 0.825,
  'risk_level': 'medio'},
 {'event_id': 'EVT-MOCK-003',
  'timestamp': '2026-05-31T12:00:00Z',
  'area_id': 'BR-MT-003',
  'source': 'FIRMS',
  'detected_class': 'agua',
  'class_percentage': 35.96,
  'change_score': 0.322,
  'cloud_score': 0.047,
  'shadow_score': 0.202,
  'image_quality': 0.914,
  'cv_confidence': 0.795,
  'risk_level': 'medio'}]

## 2. Pre-processamento

O pre-processamento separa treino/teste, normaliza variaveis numericas pela escala do treino e aplica one-hot encoding para `detected_class` e `source`. A escala de `class_percentage` permanece 0-100 no contrato, mas vira 0-1 na matriz de features.

In [2]:
NUMERIC_FEATURES = (
    "class_percentage",
    "change_score",
    "cloud_score",
    "shadow_score",
    "image_quality",
    "cv_confidence",
)


def split_train_test(rows, test_ratio=0.30):
    shuffled = list(rows)
    Random(42).shuffle(shuffled)
    split_at = int(len(shuffled) * (1 - test_ratio))
    return shuffled[:split_at], shuffled[split_at:]


def fit_minmax(rows):
    stats = {}
    for feature in NUMERIC_FEATURES:
        values = [row[feature] for row in rows]
        stats[feature] = (min(values), max(values))
    return stats


def scale_value(value, minimum, maximum):
    if maximum == minimum:
        return 0.0
    return (value - minimum) / (maximum - minimum)


def vectorize(row, stats):
    numeric = [scale_value(row[feature], *stats[feature]) for feature in NUMERIC_FEATURES]
    class_flags = [1.0 if row["detected_class"] == value else 0.0 for value in DETECTED_CLASSES]
    source_flags = [1.0 if row["source"] == value else 0.0 for value in SOURCES]
    return numeric + class_flags + source_flags


train_rows, test_rows = split_train_test(dataset)
scaler = fit_minmax(train_rows)
x_train = [vectorize(row, scaler) for row in train_rows]
y_train = [row["risk_level"] for row in train_rows]
x_test = [vectorize(row, scaler) for row in test_rows]
y_test = [row["risk_level"] for row in test_rows]

print(f"treino: {len(train_rows)} | teste: {len(test_rows)} | features: {len(x_train[0])}")

treino: 50 | teste: 22 | features: 15


## 3. Modelo simples

O modelo abaixo e um classificador por centroide. Ele aprende um vetor medio para cada classe de risco no conjunto de treino e prediz a classe cujo centroide fica mais perto da amostra de teste. E simples, explicavel e suficiente para evidenciar o fluxo ML do MVP.

In [3]:
def mean_vector(vectors):
    return [sum(values) / len(values) for values in zip(*vectors)]


def train_centroid_classifier(features, labels):
    centroids = {}
    for risk_level in RISK_LEVELS:
        vectors = [vector for vector, label in zip(features, labels) if label == risk_level]
        if vectors:
            centroids[risk_level] = mean_vector(vectors)
    return centroids


def distance(left, right):
    return sqrt(sum((a - b) ** 2 for a, b in zip(left, right)))


def predict_one(centroids, features):
    return min(centroids, key=lambda label: distance(features, centroids[label]))


model = train_centroid_classifier(x_train, y_train)
predictions = [predict_one(model, features) for features in x_test]
print("centroides treinados:", sorted(model.keys()))

centroides treinados: ['alto', 'baixo', 'medio']


## 4. Metricas minimas

As metricas incluem acuracia, matriz de confusao e precision/recall por nivel de risco. Para entrega academica, isso mostra desempenho minimo sem transformar o notebook em pipeline de producao.

In [4]:
def confusion_matrix(labels, predicted):
    matrix = {actual: {pred: 0 for pred in RISK_LEVELS} for actual in RISK_LEVELS}
    for actual, pred in zip(labels, predicted):
        matrix[actual][pred] += 1
    return matrix


def safe_divide(numerator, denominator):
    return 0.0 if denominator == 0 else numerator / denominator


def classification_report(labels, predicted):
    matrix = confusion_matrix(labels, predicted)
    report = {}
    for risk_level in RISK_LEVELS:
        true_positive = matrix[risk_level][risk_level]
        predicted_total = sum(matrix[actual][risk_level] for actual in RISK_LEVELS)
        actual_total = sum(matrix[risk_level].values())
        report[risk_level] = {
            "precision": round(safe_divide(true_positive, predicted_total), 3),
            "recall": round(safe_divide(true_positive, actual_total), 3),
        }
    return report


accuracy = safe_divide(
    sum(1 for actual, pred in zip(y_test, predictions) if actual == pred),
    len(y_test),
)
matrix = confusion_matrix(y_test, predictions)
report = classification_report(y_test, predictions)

print(f"acuracia: {accuracy:.3f}")
print("matriz de confusao:")
for actual in RISK_LEVELS:
    print(actual, matrix[actual])
print("precision/recall:", report)

acuracia: 0.727
matriz de confusao:
baixo {'baixo': 3, 'medio': 0, 'alto': 0}
medio {'baixo': 3, 'medio': 9, 'alto': 1}
alto {'baixo': 0, 'medio': 2, 'alto': 4}
precision/recall: {'baixo': {'precision': 0.5, 'recall': 1.0}, 'medio': {'precision': 0.818, 'recall': 0.692}, 'alto': {'precision': 0.8, 'recall': 0.667}}


## 5. Conclusao

O experimento comprova o caminho minimo de Machine Learning para o Orbital Trust: dataset controlado no contrato IoT, pre-processamento reprodutivel, modelo treinado, metricas e conclusao sem dependencia de webcam ou servicos privados.

Para o MVP, o classificador por centroides e apenas uma baseline academica. Em evolucoes futuras, ele pode ser substituido por `scikit-learn` com dados historicos reais, mantendo os mesmos campos de entrada e a mesma resposta `risk_level` para o mobile.